In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import gc
from scipy.stats import pearsonr
import polars as pl
import lightgbm as lgb
import numpy as np
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold
from joblib import Parallel, delayed


In [ ]:
# Load data
train_data = pl.read_parquet("/kaggle/input/drw-crypto-market-prediction/train.parquet")
test_data = pl.read_parquet("/kaggle/input/drw-crypto-market-prediction/test.parquet")
sample_submission = pl.read_csv("/kaggle/input/drw-crypto-market-prediction/sample_submission.csv")

In [ ]:
import pickle
with open('/kaggle/input/select-features/selected_features.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

In [ ]:
# Convert to numpy arrays
X_train = train_data[feature_cols].to_numpy()
#X_train = train_data.drop("label").to_numpy()
y_train = train_data["label"].to_numpy()

# Remove NaN values from y_train
valid_mask = ~np.isnan(y_train)
X_train = X_train[valid_mask]
y_train = y_train[valid_mask]

# Split training data into train and validation sets
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

X_test = test_data[feature_cols].to_numpy()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Step 1: Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)     # fit + transform on train
X_val_scaled = scaler.transform(X_val)             # transform only on val
X_test_scaled = scaler.transform(X_test)           # transform only on test

# Step 2: Apply PCA
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)    # fit + transform on train
X_val_pca = pca.transform(X_val_scaled)            # transform only on val
X_test_pca = pca.transform(X_test_scaled)          # transform only on test


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# L2 正则化：解决共线性问题
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10], cv=5)

# L1 正则化：自动筛选不重要特征（feature selection）
lasso = LassoCV(alphas=[0.01, 0.1, 1], cv=5, max_iter=10000)

# 综合 L1 + L2
elastic = ElasticNetCV(l1_ratio=[.1, .5, .9], alphas=[0.01, 0.1, 1], cv=5)


ridge = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5)
ridge.fit(X_train_pca, y_train)
print("Best alpha:", ridge.alpha_)
print("Validation IC:", np.corrcoef(y_val, ridge.predict(X_val_pca))[0,1])

lasso = LassoCV(alphas=[0.1, 1.0, 10.0], cv=5)
lasso.fit(X_train_pca, y_train)
print("Best alpha:", lasso.alpha_)
print("Validation IC:", np.corrcoef(y_val, lasso.predict(X_val_pca))[0,1])

elastic = ElasticNetCV(alphas=[0.1, 1.0, 10.0], cv=5)
elastic.fit(X_train_pca, y_train)
print("Best alpha:", elastic.alpha_)
print("Validation IC:", np.corrcoef(y_val, elastic.predict(X_val_pca))[0,1])


In [ ]:
predictions = ridge.predict(X_test_pca )
n_predictions = len(X_test_pca )
# Create submission file
submission = pd.DataFrame({
    "ID": range(1, n_predictions + 1),
    "prediction": predictions
})
submission.head()


In [ ]:
submission.to_csv("submission.csv", index=False)